# Setup

In [5]:
# !pip install selenium webdriver-manager pandas beautifulsoup4 lxml

In [6]:
import os
from dotenv import load_dotenv
import time
import pandas as pd
import re
import json
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

from webdriver_manager.chrome import ChromeDriverManager

# Config

In [7]:
load_dotenv()

BASE_URL = "https://haims2.doh.go.th/"
USERNAME = os.getenv("USERNAME")
PASSWORD = os.getenv("PASSWORD")

TARGET_INCIDENT_ID = "1158950"

WAIT_TIME = 10
EXPORT_FORMAT = "JSON"

## Google Drive Config

In [8]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive

gauth = GoogleAuth()
gauth.LocalWebserverAuth() 
drive = GoogleDrive(gauth)

MAIN_GDRIVE_FOLDER_ID = '126ltPgBFcu-AZjWCjErDRpLPlB2-mEKx'
print("✅ Connected Google Drive successfully!")

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?client_id=839515928127-ccb442os010d6bb2qjmh43n0s478rhg1.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8090%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&access_type=online&response_type=code

Authentication successful.
✅ Connected Google Drive successfully!


# Start Browser

In [9]:
options = Options()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

wait = WebDriverWait(driver, WAIT_TIME)

driver.get(BASE_URL)

# Login

In [10]:
wait.until(lambda d: d.execute_script("return document.readyState") == "complete")

username_input = wait.until(
    EC.element_to_be_clickable((By.ID, "edit-name"))
)

password_input = wait.until(
    EC.element_to_be_clickable((By.ID, "edit-pass"))
)

login_button = wait.until(
    EC.element_to_be_clickable((By.ID, "edit-submit"))
)

username_input.clear()
username_input.click()
username_input.send_keys(USERNAME)

password_input.clear()
password_input.click()
password_input.send_keys(PASSWORD)

time.sleep(5)

login_button.click()

# time.sleep(3)

wait.until(
    EC.presence_of_element_located((By.ID, "summary_main_table"))
)

print("Login Success")

Login Success


# Helper Function

In [11]:
html = driver.page_source
soup = BeautifulSoup(html,"html.parser")

### Universal Element Finder

In [12]:
def find_element(key, tag=None):

    # search by name
    el = soup.find(tag or True, {"name": key})
    if el:
        return el

    # search by id
    el = soup.find(tag or True, {"id": key})
    if el:
        return el

    # search by class
    el = soup.find(tag or True, class_=key)
    if el:
        return el

    # search label tag
    label = soup.find("label", string=lambda x: x and key in x)
    if label:
        return label.find_next()

    # search text label (span/div)
    text_label = soup.find(string=lambda x: x and key in x)
    if text_label:
        return text_label.parent

    return None

### Universal Value Extractor

In [13]:
def get_value(el):

    if not el:
        return None

    # ถ้ามี input อยู่ข้างใน
    inputs = el.find_elements(By.TAG_NAME, "input")
    if inputs:
        return inputs[0].get_attribute("value")

    # textarea
    textareas = el.find_elements(By.TAG_NAME, "textarea")
    if textareas:
        return textareas[0].get_attribute("value")

    # select
    selects = el.find_elements(By.TAG_NAME, "select")
    if selects:
        for opt in selects[0].find_elements(By.TAG_NAME, "option"):
            if opt.is_selected():
                return opt.text.strip()

    return el.text.strip()

In [14]:
def get_attr_by_id(element_id, attr):
    try:
        el = driver.find_element(By.ID, element_id)
        return el.get_attribute(attr)
    except:
        return None

### Universal Driver Element Finder

In [15]:
def get_by_key(key):
    
    try:
        el = driver.find_element(By.NAME, key)
        return el.get_attribute("value")
    except:
        pass

    try:
        el = driver.find_element(By.NAME, key)
        return get_value(el)
    except:
        pass

    try:
        el = driver.find_element(By.ID, key)
        return get_value(el)
    except:
        pass

    try:
        el = driver.find_element(By.CLASS_NAME, key)
        return get_value(el)
    except:
        pass

    return None

In [16]:
def get_by_css(css):
    el = driver.find_element(By.CSS_SELECTOR, css)
    return get_value(el)

### Checkbox & Label Finder

In [17]:
def get_checkbox_labels(name):

    results = []

    checkboxes = driver.find_elements(By.NAME, name)

    for cb in checkboxes:
        if cb.is_selected():

            cb_id = cb.get_attribute("id")

            label = driver.find_element(By.CSS_SELECTOR, f"label[for='{cb_id}']")
            results.append(label.text.strip())

    return results

# Extract

## SECTION 0: Header

In [18]:
def split_user_datetime(text):

    if not text:
        return None, None

    parts = text.rsplit(" ", 4)

    if len(parts) >= 5:
        user = parts[0]
        dt = " ".join(parts[1:])
        return user.strip(), dt.strip()

    return None, None

In [19]:
def section_0(project_id):
    userdata = driver.find_element(By.ID, "userdata").text
    lines = userdata.split("\n")

    create_text = lines[0].replace("สร้าง:", "").strip()
    update_text = lines[1].replace("แก้ไข:", "").strip()

    create_user, create_datetime = split_user_datetime(create_text)
    update_user, update_datetime = split_user_datetime(update_text)
    
    metadata = driver.find_elements(By.CSS_SELECTOR, "#metadata .d-inline-block")

    highway_reporter = get_value(metadata[0])
    receiver = get_value(metadata[1])
    sender = get_value(metadata[2])
    
    # ------ Gathering Parts ------
    
    header = {
        "project_id": project_id,
        "create_datetime": create_datetime,
        "create_user": create_user,
        "update_datetime": update_datetime,
        "update_user": update_user,
        "highway_reporter": highway_reporter,
        "receiver": receiver,
        "sender": sender,
    }

    header_df = pd.DataFrame([header])
    
    return header_df

## SECTION 1

In [20]:
def section_1(project_id):
    date = driver.find_element(By.ID,"date-mask").get_attribute("value")
    hour = driver.find_element(By.ID,"hour").find_element(By.CSS_SELECTOR,"option:checked").text
    minute = driver.find_element(By.ID,"minute").find_element(By.CSS_SELECTOR,"option:checked").text
    
    section_1_df = pd.DataFrame([{
        "project_id": project_id,
        "incident_datetime": f"{date} {hour}:{minute}"
    }])
    return section_1_df

## SECTION 2

In [21]:
def section_2(project_id):
    section_2_df = pd.DataFrame([{
        "project_id": project_id,
        "is_major_accident": 1 if driver.find_element(By.ID,"is_big_accident").is_selected() else 0,
        "is_public_interest": 1 if driver.find_element(By.ID,"has_public_figure").is_selected() else 0
    }])
    return section_2_df

## SECTION 3

In [23]:
def section_3(project_id):
    is_new_construction_zone = 1 if driver.find_element(By.ID,"is_new_construct_road").is_selected() else 0

    highway_district = driver.find_element(By.ID, "bureau") \
        .find_element(By.CSS_SELECTOR, "option[selected]").text
    highway_subdistrict = driver.find_element(By.ID, "district").get_attribute("value")
    highway_maintenance_unit = driver.find_element(By.ID, "depo").get_attribute("value")

    highway_number = driver.find_element(By.ID, "route").get_attribute("value")
    control_section = driver.find_element(By.ID, "controlsection").get_attribute("value")
    kilopost = driver.find_element(By.ID, "kilometre").get_attribute("value").lstrip('0') + "+" + driver.find_element(By.ID, "metre").get_attribute("value")

    roadinfo_text = driver.find_element(By.ID, "roadinfo").text
    parts = [x.strip() for x in roadinfo_text.split(" / ")]
    road_name = None
    section_name = None
    province = None
    section_start_km = None
    section_end_km = None
    if len(parts) > 0:
        km_text = parts[-1] 
        
        km = re.findall(r'\d+\+\d+', km_text)
        if len(km) >= 2:
            section_start_km = km[0]
            section_end_km = km[1]
        elif len(km) == 1:
            section_start_km = km[0]
            
        if len(parts) >= 4:
            road_name = parts[0]
            section_name = parts[1]
            province = parts[-2] 
            
        elif len(parts) == 3:
            road_name = parts[0]
            province = parts[1]
            
        elif len(parts) == 2:
            road_name = parts[0]

    road_feature = driver.find_element(By.ID, "road_char").text
    road_status = driver.find_element(By.ID, "road_condition").text
    lanes_count = driver.find_element(By.ID, "road_lane").get_attribute("value")

    road_direction = driver.find_element(By.ID, "road_direction").text
    median_type = driver.find_element(By.ID, "road_isle").text
    traffic_type = driver.find_element(By.ID, "road_traffic").text
    surface_type = driver.find_element(By.ID, "road_surface").text
    
    # ------ Gathering Parts ------
    
    section_3 = {
        "project_id": project_id,

        "is_new_construction_zone": is_new_construction_zone,

        "highway_district": highway_district,
        "highway_subdistrict": highway_subdistrict,
        "highway_maintenance_unit": highway_maintenance_unit,

        "highway_number": highway_number,
        "control_section": control_section,
        "kilopost": kilopost,

        "road_name": road_name,
        "section_name": section_name,
        "province": province,
        "section_start_km": section_start_km,
        "section_end_km": section_end_km,

        "road_feature": road_feature,
        "road_status": road_status,
        "lanes_count": lanes_count,

        "direction": road_direction,
        "median_type": median_type,
        "traffic_type": traffic_type,
        "surface_type": surface_type,
    }

    section_3_df = pd.DataFrame([section_3])
    
    return section_3_df

## SECTION 4

In [24]:
def section_4(project_id):
    
    road_horizontal = driver.find_element(By.ID, "char_horizontal").text
    road_vertical = driver.find_element(By.ID, "char_vertical").text
    intersection = driver.find_element(By.ID, "char_intersection").text
    median_opening = driver.find_element(By.ID, "char_openacess").text
    connector = driver.find_element(By.ID, "char_connect").text
    special_area = driver.find_element(By.ID, "char_other").text
    
    # ------ Gathering Parts ------
    
    section_4 = {
        "project_id": project_id,
        "road_horizontal": road_horizontal,
        "road_vertical": road_vertical,
        "intersection": intersection,
        "median_opening": median_opening,
        "connector": connector,
        "special_area": special_area,
    }

    section_4_df = pd.DataFrame([section_4])
    
    return section_4_df

## SECTION 5

In [25]:
def get_span_section_5():

    try:
        control_span = wait.until(EC.presence_of_element_located((By.ID, "control")))
        
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", control_span)
        time.sleep(1) 
        
        driver.execute_script("arguments[0].click();", control_span)
        print("Clicked on 'control' span successfully.")
        
        time.sleep(2) 
        
    except Exception as e:
        print(f"Cannot click the control span: {e}")

In [26]:
def section_5(project_id):
    
    get_span_section_5()
    
    selected_controls = get_checkbox_labels("control[]")
    
    # ------ Gathering Parts ------
    
    section_5 = {
        "project_id": project_id,
        "speed_sign": 1 if "ป้ายจำกัดความเร็ว" in selected_controls else 0,
        "stop_sign": 1 if "ป้ายบังคับหยุด" in selected_controls else 0,
        "warning_sign": 1 if "ป้ายจราจรประเภทเตือนอื่นๆ" in selected_controls else 0,
        "traffic_light": 1 if "สัญญาณไฟจราจร" in selected_controls else 0,
        "flashing_light": 1 if "สัญญาณไฟกระพริบ" in selected_controls else 0,
        "road_marking": 1 if "เส้นเครื่องหมายจราจรบนผิวทาง" in selected_controls else 0,
        "no_overtake_zone": 1 if "เขตห้ามแซง" in selected_controls else 0,
        "no_parking_zone": 1 if "เขตห้ามจอด" in selected_controls else 0,
        "uncontrolled_crosswalk": 1 if "เป็นทางคนเดินข้ามถนนที่ไม่มีการควบคุม" in selected_controls else 0,
        "controlled_crosswalk": 1 if "เป็นทางคนเดินข้ามถนนที่มีการควบคุม" in selected_controls else 0,
        "pedestrian_bridge": 1 if "สะพานลอยคนเดินข้าม" in selected_controls else 0,
        "optical_speed_bar": 1 if "Optical Speed Bar" in selected_controls else 0,
        "rumble_strip": 1 if "Rumble Strip" in selected_controls else 0,
        "speed_camera": 1 if "กล้องตรวจจับความเร็ว" in selected_controls else 0,
        "guide_sign": 1 if "เครื่องหมายนำทาง" in selected_controls else 0,
        "no_control": 1 if "ไม่มีการควบคุมอย่างใดอย่างหนึ่ง" in selected_controls else 0,
    }

    section_5_df = pd.DataFrame([section_5])
    
    return section_5_df

## SECTION 6

In [27]:
def get_time_dropdown(key):
    """ฟังก์ชันเฉพาะสำหรับดึง Text จาก Dropdown (Select)"""
    try:
        try:
            el = driver.find_element(By.ID, key)
        except:
            el = driver.find_element(By.NAME, key)
            
        select = Select(el)
        return select.first_selected_option.text.strip()
    except:
        return ""

In [28]:
def const_section_6():
    traffic_map = {
        "1": "น้อยกว่า 500 เมตร",
        "2": "500 เมตร ถึง 1 กิโลเมตร",
        "3": "1 กิโลเมตร ถึง 2 กิโลเมตร",
        "4": "มากกว่า 2 กิโลเมตร",
        "5": "ไม่ติดสะสม"
    }
    
    return traffic_map

In [29]:
def section_6(project_id):
    traffic_map = const_section_6()
    traffic_congestion_no = get_attr_by_id("traffic_condition_parent", "data-value")
    
    start_hour = get_time_dropdown("traffic_condition_begin-hour")
    start_min  = get_time_dropdown("traffic_condition_begin-minute")
    end_hour   = get_time_dropdown("traffic_condition_end-hour")
    end_min    = get_time_dropdown("traffic_condition_end-minute")
    
    section_6_df = pd.DataFrame([{
        "project_id": project_id,
        "traffic_congestion": traffic_map.get(traffic_congestion_no),
        "start_time": f"{start_hour}:{start_min}",
        "end_time": f"{end_hour}:{end_min}",
    }])
    return section_6_df

## SECTION 7

In [30]:
def section_7(project_id):
    section_7_df = pd.DataFrame([{
        "project_id": project_id,
        "road_surface": driver.find_element(By.ID, "env_surface").text,
        "road_condition": driver.find_element(By.ID, "env_status").text,
        "weather": driver.find_element(By.ID, "env_weather").text,
        "lighting": driver.find_element(By.ID, "env_light").text,
    }])
    return section_7_df

## SECTION 8

In [32]:
def const_section_8():
    gender_map = {
        "1": "ชาย",
        "2": "หญิง"
    }

    safety_map = {
        "1": "หมวกนิรภัย",
        "2": "เข็มขัดนิรภัย",
        "3": "ไม่ใช้",
        "4": "ไม่ระบุ"
    }

    injury_map = {
        "1": "ตาย ณ จุดเกิดเหตุ",
        "2": "ตาย ณ โรงพยาบาล",
        "3": "บาดเจ็บสาหัส",
        "4": "บาดเจ็บเล็กน้อย",
        "5": "ไม่ได้รับบาดเจ็บ",
        "6": "ไม่ระบุ"
    }

    drug_map = {
        "1": "มี",
        "2": "ไม่มี",
        "3": "ไม่ระบุ"
    }
    
    return gender_map, safety_map, injury_map, drug_map

In [33]:
def section_8(project_id):
    
    gender_map, safety_map, injury_map, drug_map = const_section_8()
    
    vehicles = []

    vehicle_blocks = soup.select("#vehiclelist li")

    for v in vehicle_blocks:

        def get_value(name):
            el = v.select_one(f'input[name="{name}"]')
            return el["value"].strip() if el else None
        
        # ------ Gathering Parts ------

        item = {
            "project_id": project_id,
            "vehicle_brand": get_value("vehicle_brand[]"),
            "color": get_value("vehicle_color[]"),
            "driver_name": get_value("vehicle_name[]"),
            "driver_citizen_id": get_value("vehicle_id_no[]"),
            "driver_age": get_value("vehicle_age[]"),
            "driver_gender": gender_map.get(get_value("vehicle_sex[]")),
            "safety_equipment": safety_map.get(get_value("vehicle_safety[]")),
            "injury_status": injury_map.get(get_value("vehicle_damage[]")),
        }

        vehicles.append(item)

    section_8_df = pd.DataFrame(vehicles)
    
    return section_8_df

## SECTION 9

In [34]:
def const_section_9():
    categories = {
        "100": "ด้านสภาพแวดล้อม",
        "200": "ด้านยานพาหนะ",
        "300": "ด้านผู้ขับขี่",
        "400": "ด้านคนเดินเท้า"
    }
    
    return categories

In [35]:
def section_9(project_id):
    
    categories = const_section_9()
    
    causes = []

    for code, category_name in categories.items():

        items = soup.select(f"#causelist-{code}-items li")

        for i, item in enumerate(items, start=1):

            cause = item.text.split("\n ")[1].strip()
            
            # ------ Gathering Parts ------

            data = {
                "project_id": project_id,
                "category": category_name,
                "rank": i,
                "cause": cause
            }

            causes.append(data)

    section_9_df = pd.DataFrame(causes)
    
    return section_9_df

## SECTION 10

In [36]:
def section_10(project_id):
    try:
        sec10_btn = wait.until(EC.presence_of_element_located((By.ID, "damage")))
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", sec10_btn)
        driver.execute_script("arguments[0].click();", sec10_btn)
        time.sleep(1) 
    except Exception as e:
        print(f"⚠️ Can NOT click for Section 10: {e}")

    soup = BeautifulSoup(driver.page_source, "html.parser")
    
    checked_labels = [cb.find_next("label").text.strip() for cb in soup.select("input[type='checkbox']:checked") if cb.find_next("label")]

    section_10_df = pd.DataFrame([{
        "project_id": project_id,
        "damage_sign": 1 if "ป้ายจราจร" in str(checked_labels) else 0,
        "damage_guardrail": 1 if "ราวกันอันตราย" in str(checked_labels) else 0,
        "damage_electric_light": 1 if "อุปกรณ์ไฟฟ้า" in str(checked_labels) else 0,
        "damage_traffic_signal": 1 if "อุปกรณ์สัญญาณ" in str(checked_labels) else 0,
        "damage_road_surface": 1 if "ผิวจราจร" in str(checked_labels) else 0,
        "damage_shoulder": 1 if "คันทาง" in str(checked_labels) else 0,
        "damage_bridge": 1 if "สะพาน" in str(checked_labels) else 0,
        "damage_km_post": 1 if "หลัก กม." in str(checked_labels) else 0,
        "damage_median": 1 if "เกาะ / อุปกรณ์กั้นกลางถนน" in str(checked_labels) else 0,
        "damage_tree": 1 if "ต้นไม้" in str(checked_labels) else 0,
        "damage_bus_stop": 1 if "ศาลาทางหลวง" in str(checked_labels) else 0,
        "damage_roundabout": 1 if "เกาะกลางในวงเวียน" in str(checked_labels) else 0,
        "damage_government_vehicle": 1 if "รถราชการ" in str(checked_labels) else 0,
        "damage_traffic_counter": 1 if "เครื่องสำรวจปริมาณจราจร" in str(checked_labels) else 0,
        "damage_reflector": 1 if "เครื่องหมายนำทาง" in str(checked_labels) else 0,
        "damage_its_equipment": 1 if "อุปกรณ์ ITS" in str(checked_labels) else 0,
        "damage_retaining_wall": 1 if "กำแพงกันดิน" in str(checked_labels) else 0,
        "damage_other": 1 if "อื่น ๆ" in str(checked_labels) else 0
    }])
    return section_10_df

## SECTION 11

In [37]:
def section_11(project_id):
    try:
        sec11_btn = wait.until(EC.presence_of_element_located((By.ID, "fatal")))
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", sec11_btn)
        driver.execute_script("arguments[0].click();", sec11_btn)
        time.sleep(1) 
    except:
        pass 

    soup = BeautifulSoup(driver.page_source, "html.parser")
    
    def get_row_values(row_element):
        if not row_element:
            return [0] * 7
        
        inputs = row_element.select("input[type='text']")
        vals = []
        for inp in inputs:
            val = inp.get('value', '').strip()
            vals.append(int(val) if val.isdigit() else 0)
            
        while len(vals) < 7:
            vals.append(0)
        return vals

    death_rows = soup.select("tr.fatal_death") 
    wound_rows = soup.select("tr.fatal_wound") 

    scene_vals = get_row_values(death_rows[0]) if len(death_rows) > 0 else [0]*7
    hospital_vals = get_row_values(death_rows[1]) if len(death_rows) > 1 else [0]*7
    severe_vals = get_row_values(wound_rows[0]) if len(wound_rows) > 0 else [0]*7
    minor_vals = get_row_values(wound_rows[1]) if len(wound_rows) > 1 else [0]*7

    section_11_df = pd.DataFrame([{
        "project_id": project_id,
        
        "death_scene_adult_male": scene_vals[0], 
        "death_scene_adult_female": scene_vals[1],
        "death_scene_child_male": scene_vals[2],
        "death_scene_child_female": scene_vals[3],
        "death_scene_unknown_male": scene_vals[4],
        "death_scene_unknown_female": scene_vals[5],
        "death_scene_unknown": scene_vals[6],

        "death_hospital_adult_male": hospital_vals[0], 
        "death_hospital_adult_female": hospital_vals[1], 
        "death_hospital_child_male": hospital_vals[2], 
        "death_hospital_child_female": hospital_vals[3], 
        "death_hospital_unknown_male": hospital_vals[4], 
        "death_hospital_unknown_female": hospital_vals[5], 
        "death_hospital_unknown": hospital_vals[6], 

        "injury_severe_adult_male": severe_vals[0], 
        "injury_severe_adult_female": severe_vals[1], 
        "injury_severe_child_male": severe_vals[2], 
        "injury_severe_child_female": severe_vals[3], 
        "injury_severe_unknown_male": severe_vals[4], 
        "injury_severe_unknown_female": severe_vals[5], 
        "injury_severe_unknown": severe_vals[6], 

        "injury_minor_adult_male": minor_vals[0], 
        "injury_minor_adult_female": minor_vals[1], 
        "injury_minor_child_male": minor_vals[2], 
        "injury_minor_child_female": minor_vals[3], 
        "injury_minor_unknown_male": minor_vals[4], 
        "injury_minor_unknown_female": minor_vals[5], 
        "injury_minor_unknown": minor_vals[6], 

        "consumable_materials": get_attr_by_id("fatal_waste_material", "value"),
        "government_damage_cost": get_attr_by_id("fatal_damage_public", "value"),
        "private_damage_cost": get_attr_by_id("fatal_damage_private", "value")
    }])
    
    return section_11_df

## SECTION 12

In [38]:
def section_12(project_id):
    soup = BeautifulSoup(driver.page_source, "html.parser")
    
    def get_construct_val(db_key_name):
        el = soup.select_one(f"#constructionDisplay span[db-key='{db_key_name}']")
        return el.text.strip() if el else None

    section_12_df = pd.DataFrame([{
        "project_id": project_id,
        
        "project_type": get_construct_val("construct_type_name"),
        "project_name": get_construct_val("construct_name"),
        "contract_number": get_construct_val("construct_contract_no"),
        "responsible_agency": get_construct_val("construct_department"),
        "project_other_details": get_construct_val("construct_other"),
        
        "construction_km_start": get_construct_val("construct_km_start"),
        "construction_km_end": get_construct_val("construct_km_end"),
        
        "accident_location_in_construction": get_construct_val("construct_accident_location_name"),
        "traffic_warning_sign_condition": get_construct_val("construct_sign_cond_name"),
        
        "traffic_queue": get_construct_val("construct_affect_name"),
        "traffic_queue_start": get_construct_val("construct_affect_begin"), 
        "traffic_queue_end": get_construct_val("construct_affect_end"),
        
        "lane_closure_management": get_construct_val("construct_solution"),
        "workzone_management": get_construct_val("managementString"),
        "staff_present_in_worksite": get_construct_val("construct_has_staff_name"),
        "assistance_provided_by": get_construct_val("construct_help_department")
    }])
    
    return section_12_df

## SECTION Diagram

In [39]:
def section_diagram(project_id):
    
    crash_id = driver.find_element(By.ID, "crash").text
    inci_address = soup.select_one("#address").get_text(strip=True).replace("สถานที่:", "")
    gps_loc = soup.select_one("#position").get_text(strip=True).split(', ')
    gps_lat = gps_loc[0]
    gps_lon = gps_loc[1]
    inci_desc = soup.select_one("#summary").get_text(strip=True)
        
    diagram = {
        "project_id": project_id,
        "crash_id": crash_id,
        "incident_address": inci_address,
        "gps_lat": gps_lat,
        "gps_lng": gps_lon,
        "incident_description": inci_desc,
    }

    diagram_df = pd.DataFrame([diagram])
    
    return diagram_df

# Save and Upload

## .csv

In [40]:
def save_and_upload_csv_direct(project_id, drive, main_folder_id, 
                                header_df, section_1_df, section_2_df, section_3_df, 
                                section_4_df, section_5_df, section_6_df, section_7_df, 
                                section_8_df, section_9_df, section_10_df, section_11_df, 
                                section_12_df, diagram_df):

    folder_name = f"{project_id}_dataset"
    print(f"☁️ Creating {folder_name} on Google Drive...")
    
    folder_metadata = {'title': folder_name, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [{'id': main_folder_id}]}
    gdrive_folder = drive.CreateFile(folder_metadata)
    gdrive_folder.Upload()
    new_folder_id = gdrive_folder['id'] 

    section_1_2_df = pd.merge(section_1_df, section_2_df, on="project_id", how="outer")
    section_6_7_df = pd.merge(section_6_df, section_7_df, on="project_id", how="outer")

    files_to_upload = {
        "header.csv": header_df,
        "section_1_2.csv": section_1_2_df, 
        "section_3.csv": section_3_df,
        "section_4.csv": section_4_df,
        "section_5.csv": section_5_df,
        "section_6_7.csv": section_6_7_df, 
        "section_8.csv": section_8_df,
        "section_9.csv": section_9_df,
        "section_10.csv": section_10_df,
        "section_11.csv": section_11_df,
        "section_12.csv": section_12_df,
        "section_diagram.csv": diagram_df
    }
    
    for filename, df in files_to_upload.items():
        csv_content = df.to_csv(index=False, encoding='utf-8-sig')
        file_metadata = {'title': filename, 'parents': [{'id': new_folder_id}]}
        file_drive = drive.CreateFile(file_metadata)
        file_drive.SetContentString(csv_content) 
        file_drive.Upload()

    print(f"🎉 Uploaded data ID: {project_id} to Google Drive successfully!\n")

## .json

In [41]:
def save_and_upload_json_direct(project_id, drive, main_folder_id, 
                                header_df, section_1_df, section_2_df, section_3_df, 
                                section_4_df, section_5_df, section_6_df, section_7_df, 
                                section_8_df, section_9_df, section_10_df, section_11_df, 
                                section_12_df, diagram_df):
    
    print(f"☁️ Structuring data in Project ID: {project_id} into JSON...")
    
    def get_single_dict(df):
        return df.to_dict('records')[0] if not df.empty else {}
    
    def get_list_dict(df):
        return df.to_dict('records') if not df.empty else []

    combined_json = {
        "project_id": project_id,
        # SECTION 0
        "header": get_single_dict(header_df),
        # SECTION 1
        "เวลาเกิดเหตุ": get_single_dict(section_1_df), 
        # SECTION 2
        "อุบัติเหตุใหญ่": get_single_dict(section_2_df), 
        # SECTION 3
        "ข้อมูลทางหลวง": get_single_dict(section_3_df),
        # SECTION 4
        "ลักษณะบริเวณที่เกิดเหตุ": get_single_dict(section_4_df), 
        # SECTION 5
        "การควบคุมการใช้ทางหลวง": get_single_dict(section_5_df),
        # SECTION 6
        "ปริมาณรถสะสมขณะเกิดเหตุ": get_single_dict(section_6_df), 
        # SECTION 7
        "ทัศนวิสัย/สภาพแวดล้อม": get_single_dict(section_7_df), 
        # SECTION 8
        "ข้อมูลเกี่ยวกับรถที่ประสบอุบัติเหตุ": get_list_dict(section_8_df), 
        # SECTION 9
        "มูลเหตุที่สันนิษฐาน": get_list_dict(section_9_df), 
        # SECTION 10
        "ทรัพย์สินเสียหาย": get_single_dict(section_10_df),
        # SECTION 11
        "ความเสียหายจากอุบัติเหตุ": get_single_dict(section_11_df),
        # SECTION 12
        "รายละเอียดงานก่อสร้าง/บำรุง": get_single_dict(section_12_df),
        # SECTION DIAGRAM
        "section_diagram": get_single_dict(diagram_df)
    }

    json_content = json.dumps(combined_json, ensure_ascii=False, indent=4)
    
    filename = f"{project_id}.json"
    file_metadata = {'title': filename, 'parents': [{'id': main_folder_id}], 'mimeType': 'application/json'}
    
    file_drive = drive.CreateFile(file_metadata)
    file_drive.SetContentString(json_content) 
    file_drive.Upload()

    print(f"🎉 Uploaded {filename} successfully!\n")

## Selector format

In [42]:
def save_and_upload(opt, project_id, drive, main_folder_id, 
                    header_df, section_1_df, section_2_df, section_3_df, 
                    section_4_df, section_5_df, section_6_df, section_7_df, 
                    section_8_df, section_9_df, section_10_df, section_11_df, 
                    section_12_df, diagram_df):
    
    match opt.upper():
        case "CSV":
            save_and_upload_csv_direct( 
                project_id=project_id, drive=drive, main_folder_id=main_folder_id,
                header_df=header_df, section_1_df=section_1_df, section_2_df=section_2_df, section_3_df=section_3_df, 
                section_4_df=section_4_df, section_5_df=section_5_df, section_6_df=section_6_df, section_7_df=section_7_df, 
                section_8_df=section_8_df, section_9_df=section_9_df, section_10_df=section_10_df, section_11_df=section_11_df, 
                section_12_df=section_12_df, diagram_df=diagram_df
            )
        case "JSON":
            save_and_upload_json_direct( 
                project_id=project_id, drive=drive, main_folder_id=main_folder_id,
                header_df=header_df, section_1_df=section_1_df, section_2_df=section_2_df, section_3_df=section_3_df, 
                section_4_df=section_4_df, section_5_df=section_5_df, section_6_df=section_6_df, section_7_df=section_7_df, 
                section_8_df=section_8_df, section_9_df=section_9_df, section_10_df=section_10_df, section_11_df=section_11_df, 
                section_12_df=section_12_df, diagram_df=diagram_df
            )
        case _:
            print(f"❌ Error: '{opt}' format does NOT available")

# Direct Link

In [ ]:
# Option 1: ID เรียงต่อเนื่อง
START_ID = 1159340
STOP_ID = 1159350

INCIDENT_IDS = [str(i) for i in range(START_ID, STOP_ID + 1)]

# Option 2: ID ไม่ได้เรียงต่อเนื่อง
# INCIDENT_IDS = ["1158950", "1158955", "1158980"]

## Main

In [44]:
for project_id in INCIDENT_IDS:

    print(f"\n--- Checking Incident ID: {project_id} ---")

    target_url = f"https://haims2.doh.go.th/form/{project_id}"
    driver.get(target_url)

    try:
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "form")))
        time.sleep(2) 
        
        print(f"✅ Found ID {project_id} let's Extract Data...")
        
        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")

        header_df = section_0(project_id)
        section_1_df = section_1(project_id)
        section_2_df = section_2(project_id)
        section_3_df = section_3(project_id) 
        section_4_df = section_4(project_id)
        section_5_df = section_5(project_id)
        section_6_df = section_6(project_id)
        section_7_df = section_7(project_id)
        section_8_df = section_8(project_id)
        section_9_df = section_9(project_id)
        section_10_df = section_10(project_id)
        section_11_df = section_11(project_id)
        section_12_df = section_12(project_id)
        diagram_df = section_diagram(project_id)
        
        save_and_upload(
            opt=EXPORT_FORMAT,
            project_id=project_id, 
            drive=drive, 
            main_folder_id=MAIN_GDRIVE_FOLDER_ID,
            header_df=header_df, 
            section_1_df=section_1_df, 
            section_2_df=section_2_df, 
            section_3_df=section_3_df, 
            section_4_df=section_4_df, 
            section_5_df=section_5_df, 
            section_6_df=section_6_df, 
            section_7_df=section_7_df, 
            section_8_df=section_8_df, 
            section_9_df=section_9_df, 
            section_10_df=section_10_df,
            section_11_df=section_11_df,
            section_12_df=section_12_df,
            diagram_df=diagram_df
        )

        print(f"💾 Saved ID: {project_id} sucessfully!")

    except TimeoutException:
        print(f"⚠️ Skipped ID: {project_id} - Form not found")
        continue 

    except Exception as e:
        print(f"❌ Error on ID: {project_id} - {e}")
        continue


--- Checking Incident ID: 1159340 ---
✅ Found ID 1159340 let's Extract Data...
Clicked on 'control' span successfully.
☁️ Structuring data in Project ID: 1159340 into JSON...
🎉 Uploaded 1159340.json successfully!

💾 Saved ID: 1159340 sucessfully!
